[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_67_Async_Parallel_Benchmark_Execution.ipynb)

# Lesson 67 — Parallel Benchmark Execution: `asyncio.gather` + `Semaphore`

**Phase 7, Lesson 3 (tentative).** Previous: L65 (Phase 7 kickoff — `agent-bench` repackaged as a real installable library with a plugin registry), L66 (real third-party plugin discovery via `importlib.metadata.entry_points()` + a real `ShellEnv` plugin package).

Today's upgrade is **speed, not features**: `agent-bench`'s `BenchmarkRunner.run()` executes every task/attempt one at a time. That was fine for an 8-task demo suite. It will not be fine once `agent-bench` scores real agents against real suites with dozens of tasks × `k` attempts each, where every attempt is a real network call to the Claude API. Today we make the runner run attempts **concurrently**, bounded by a `Semaphore` — the same shape as L57's `batch_distill()` for `paper-distiller`, now applied to the benchmark harness itself.

### Phase 7 roadmap (tentative — adaptive, not a contract)

| # | Lesson | Status |
|---|--------|--------|
| 65 | Phase 7 Kickoff — `agent-bench` as a second flagship OSS tool (plugin registry pattern) | ✅ done |
| 66 | Real Plugins — `ShellEnv` + `importlib.metadata.entry_points()` | ✅ done |
| **67** | **Parallel Benchmark Execution — `asyncio.gather` + `Semaphore`** | **← today** |
| 68 | CLI polish + PyPI packaging for `agent-bench` | tentative |
| 69 | OSS Growth for `agent-bench` (README, badges, contributor funnel) | tentative |
| 70 | Phase 7 capstone / launch | tentative |


## Concept: sequential vs. concurrent benchmark execution

`BenchmarkRunner.run(agent, k=3)` on an 8-task suite makes `8 × 3 = 24` calls to `agent.act(...)`. If each call is a live Claude API round-trip (typically hundreds of milliseconds to a few seconds), running them one after another means the *harness itself* — not the model, not the task — becomes the bottleneck. That's the same problem L57 solved for `paper-distiller`'s batch mode: many independent, slow, I/O-bound calls that don't depend on each other's results.

| | Sequential `run()` | Async `run_async()` |
|---|---|---|
| Shape | `for task in tasks: for attempt in range(k): agent.act(...)` | `asyncio.gather(*coros)` bounded by `Semaphore(concurrency)` |
| Wall-clock for N calls @ latency L | `N × L` | `≈ ceil(N / concurrency) × L` |
| Failure handling | one exception can halt the whole suite | must isolate per-attempt (L57 pitfall #4: `gather` without `return_exceptions=True` fails fast) |
| New risk introduced today | none | **shared mutable state** — if two attempts run "at the same time" against the *same* environment instance, they can contaminate each other |

The new risk is the one worth slowing down for. `ENVIRONMENT_REGISTRY` in L65/L66 stores one instantiated environment object per name (`"calc" -> CalcEnv()`, `"file" -> FileEnv()`). That was safe under sequential execution because only one task ever touched an environment at a time. It is **not** safe under concurrency. We'll reproduce the bug for real before fixing it — same house rule as every lesson since L57: don't just describe a pitfall, trigger it.


In [ ]:
# Setup
!pip install anthropic pydantic rich nest_asyncio typer -q

import asyncio, time, copy, os, sys, json
from dataclasses import dataclass, field
from typing import Any, Optional, Callable
from pydantic import BaseModel, Field
from rich.console import Console
from rich.table import Table
from rich.progress import Progress, BarColumn, TextColumn, TimeElapsedColumn

import nest_asyncio
nest_asyncio.apply()

console = Console(force_jupyter=False, no_color=True, highlight=False)

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')

HAVE_API_KEY = bool(ANTHROPIC_API_KEY)
print(f"HAVE_API_KEY = {HAVE_API_KEY}  (today's lesson runs fully offline either way — no live agent needed to demonstrate concurrency)")


## Recreating `core.py` and `registry.py`

Fresh runtime each lesson (Colab-ready constraint), so we recreate the pieces we need. `core.py`'s data models are unchanged from L61/L65. `registry.py` gets **one real change today**: environments are registered as **classes (factories)**, not instances — `get_environment(name)` now returns a fresh object on every call. This is the fix for the bug we're about to demonstrate; we're showing it here so you see the *shape* before the *reason*, then we'll break the old shape on purpose in the next cell to prove the reason.


In [ ]:
# core.py — Task / Trajectory / TaskResult (unchanged shape from L61/L65)

class Task(BaseModel):
    id: str
    category: str
    difficulty: str = "medium"
    prompt: str
    env_name: str
    scorer_name: str

@dataclass
class TrajectoryStep:
    action: str
    result: Any

@dataclass
class Trajectory:
    task_id: str
    steps: list = field(default_factory=list)
    final_state: Any = None

@dataclass
class TaskResult:
    task_id: str
    attempt: int
    passed: bool
    trajectory: Optional[Trajectory]
    error: Optional[str]
    elapsed: float

def pass_at_k(n: int, c: int, k: int) -> float:
    # Unbiased pass@k estimator (Chen et al. 2021).
    if n - c < k:
        return 1.0
    import math
    return 1.0 - math.prod((n - c - i) / (n - i) for i in range(k))

print("core.py recreated: Task, TrajectoryStep, Trajectory, TaskResult, pass_at_k")


In [ ]:
# registry.py — registries store CLASSES for environments (factories), not instances.
# Agents and scorers are still fine as shared singletons/functions: MockAgent/ClaudeToolAgent
# hold no per-task mutable state, and scorer functions are pure. Environments are the one
# category that holds mutable state *by design* (a virtual filesystem, a calculator's memory,
# a corpus), so they are the one category that must never be shared across concurrent attempts.

ENVIRONMENT_REGISTRY: dict[str, type] = {}
AGENT_REGISTRY: dict[str, Any] = {}
SCORER_REGISTRY: dict[str, Callable] = {}

def register_environment(name: str):
    def deco(cls):
        if name in ENVIRONMENT_REGISTRY:
            raise ValueError(f"Environment '{name}' already registered")
        ENVIRONMENT_REGISTRY[name] = cls
        return cls
    return deco

def register_agent(name: str):
    def deco(obj):
        if name in AGENT_REGISTRY:
            raise ValueError(f"Agent '{name}' already registered")
        AGENT_REGISTRY[name] = obj
        return obj
    return deco

def register_scorer(name: str):
    def deco(fn):
        if name in SCORER_REGISTRY:
            raise ValueError(f"Scorer '{name}' already registered")
        SCORER_REGISTRY[name] = fn
        return fn
    return deco

def get_environment(name: str):
    # Returns a FRESH instance every call -- this is the fix, not an optimization.
    if name not in ENVIRONMENT_REGISTRY:
        raise KeyError(f"No environment registered as '{name}'")
    return ENVIRONMENT_REGISTRY[name]()

def get_agent(name: str):
    return AGENT_REGISTRY[name]

def get_scorer(name: str):
    return SCORER_REGISTRY[name]

print("registry.py recreated: ENVIRONMENT_REGISTRY now stores classes; get_environment() instantiates fresh each call")


In [ ]:
# Built-in environments, agent, scorers — condensed from L61/L65, with a simulated
# `latency` parameter standing in for a real Claude API round-trip. This lets us measure
# concurrency speedups deterministically without needing ANTHROPIC_API_KEY.

LATENCY = 0.15  # seconds — stand-in for one real Claude tool-call round trip

@register_environment("calc")
class CalcEnv:
    def __init__(self):
        self.history = []

    def step(self, expr: str):
        val = eval(expr, {"__builtins__": {}}, {})
        self.history.append((expr, val))
        return val

    def final_state(self):
        return self.history[-1][1] if self.history else None

@register_environment("file")
class FileEnv:
    # Virtual in-memory filesystem -- the mutable-state environment we'll use to
    # reproduce the concurrency bug, because contamination here is easy to see:
    # a task's final_state() should contain ONLY the files IT wrote.
    def __init__(self):
        self.files: dict[str, str] = {}

    def write(self, path: str, content: str):
        self.files[path] = content

    def read(self, path: str):
        return self.files.get(path)

    def final_state(self):
        return dict(self.files)

class MockAgent:
    # Deterministic scripted agent with simulated network latency and an optional
    # flake_rate, same shape as L61.
    def __init__(self, flake_rate: float = 0.0, latency: float = LATENCY):
        self.flake_rate = flake_rate
        self.latency = latency

    def act(self, task: Task, env) -> Trajectory:
        import random
        time.sleep(self.latency)  # <-- blocking, sync "network call" — this is why
                                  #     the async runner must offload it to a thread
        if random.random() < self.flake_rate:
            raise RuntimeError("simulated transient API error")
        if task.env_name == "calc":
            expr = task.prompt.split("Compute: ")[-1].split(" = ")[0]
            val = env.step(expr)
            traj = Trajectory(task_id=task.id, steps=[TrajectoryStep("calc", val)], final_state=val)
        elif task.env_name == "file":
            fname = f"{task.id}.txt"
            env.write(fname, f"result-for-{task.id}")
            traj = Trajectory(task_id=task.id, steps=[TrajectoryStep("write", fname)], final_state=env.final_state())
        else:
            traj = Trajectory(task_id=task.id, steps=[], final_state=None)
        return traj

def scorer_exact_match(trajectory: Trajectory, task: Task) -> bool:
    expected = float(task.prompt.split("=")[-1].strip()) if "=" in task.prompt else None
    return expected is not None and trajectory.final_state == expected

def scorer_file_written(trajectory: Trajectory, task: Task) -> bool:
    fname = f"{task.id}.txt"
    return isinstance(trajectory.final_state, dict) and fname in trajectory.final_state

register_scorer("exact_match")(scorer_exact_match)
register_scorer("file_written")(scorer_file_written)

print(f"Built-ins registered: environments={list(ENVIRONMENT_REGISTRY)}, scorers={list(SCORER_REGISTRY)}")


## Reproducing the bug: a shared environment instance under concurrency

Before today, `BenchmarkRunner` fetched an environment once (`ENVIRONMENT_REGISTRY["file"]`, a singleton instance) and reused it across every task, because tasks ran one at a time and each finished before the next began. Let's simulate what happens if we do that **while running attempts concurrently**: four "tasks" each write their own file to what they believe is a private filesystem, with a short `await asyncio.sleep()` standing in for the agent "thinking" between the write and the final read — enough of a gap for other coroutines to run in between on the same event loop.


In [ ]:
# THE BUG — one shared FileEnv() instance reused across concurrent attempts.

shared_env_bad = FileEnv()  # a singleton, the pre-L67 pattern

async def run_task_bad(task_id: int):
    env = shared_env_bad                      # <-- everyone gets the SAME object
    env.write(f"task_{task_id}.txt", f"content_{task_id}")
    await asyncio.sleep(0.05)                 # "agent is thinking" — yields control
    return task_id, env.final_state()

results_bad = await asyncio.gather(*[run_task_bad(i) for i in range(4)])

contaminated = False
for task_id, state in results_bad:
    own_file = f"task_{task_id}.txt"
    other_files = [f for f in state if f != own_file]
    if other_files:
        contaminated = True
    print(f"task {task_id}: expected only [{own_file}], got {sorted(state.keys())}"
          f"{'  <-- CONTAMINATED' if other_files else ''}")

assert contaminated, "expected the shared-instance bug to reproduce contamination"
print("\nConfirmed: every task's 'final_state' leaked every OTHER concurrently-running task's files.")
print("This is exactly the failure mode that would silently corrupt real benchmark scores.")


In [ ]:
# THE FIX — get_environment("file") returns a FRESH instance per call (from the
# registry.py cell above). Same coroutine shape, same interleaving, zero contamination.

async def run_task_good(task_id: int):
    env = get_environment("file")             # <-- fresh FileEnv() every call
    env.write(f"task_{task_id}.txt", f"content_{task_id}")
    await asyncio.sleep(0.05)
    return task_id, env.final_state()

results_good = await asyncio.gather(*[run_task_good(i) for i in range(4)])

for task_id, state in results_good:
    own_file = f"task_{task_id}.txt"
    assert list(state.keys()) == [own_file], f"task {task_id} leaked state: {state}"
    print(f"task {task_id}: {list(state.keys())}  <-- isolated, correct")

print("\nFixed: registering a CLASS and instantiating fresh per attempt eliminates cross-task contamination.")


## `AsyncBenchmarkRunner`: `Semaphore`-bounded concurrent execution

Now the real upgrade to the harness. `run_async(agent, k, concurrency)` builds one coroutine per `(task, attempt)` pair, bounds how many run at once with an `asyncio.Semaphore(concurrency)`, and — because `MockAgent.act()` and `ClaudeToolAgent.act()` are **synchronous, blocking** calls (a real `time.sleep()` standing in for a real blocking HTTP call) — offloads each call to a thread via `loop.run_in_executor(None, ...)` so it doesn't freeze the whole event loop. Each attempt gets its own fresh environment via `get_environment()`, so today's bugfix is baked into the runner, not something the caller has to remember.


In [ ]:
class AsyncBenchmarkRunner:
    def __init__(self, tasks: list[Task]):
        self.tasks = tasks

    async def _run_one(self, agent, task: Task, attempt: int, sem: asyncio.Semaphore, loop) -> TaskResult:
        async with sem:                                   # <-- bounds concurrency
            env = get_environment(task.env_name)           # fresh instance every attempt
            scorer = get_scorer(task.scorer_name)
            start = time.perf_counter()
            try:
                # agent.act is sync/blocking -> run it in the default ThreadPoolExecutor
                # so it doesn't block other coroutines waiting on the event loop
                trajectory = await loop.run_in_executor(None, agent.act, task, env)
                passed = scorer(trajectory, task)
                return TaskResult(task.id, attempt, passed, trajectory, None, time.perf_counter() - start)
            except Exception as e:
                # per-attempt isolation: one failure becomes a failed TaskResult,
                # not an exception that propagates out of gather() (L57 pitfall #4)
                return TaskResult(task.id, attempt, False, None, str(e), time.perf_counter() - start)

    async def run_async(self, agent, k: int = 1, concurrency: int = 4) -> list[TaskResult]:
        sem = asyncio.Semaphore(concurrency)
        loop = asyncio.get_event_loop()
        coros = [self._run_one(agent, task, a, sem, loop)
                 for task in self.tasks for a in range(k)]
        return await asyncio.gather(*coros, return_exceptions=False)  # exceptions already caught in _run_one

    def run_sync(self, agent, k: int = 1) -> list[TaskResult]:
        # Baseline: the pre-L67 sequential runner, kept for the timing comparison below.
        results = []
        for task in self.tasks:
            for attempt in range(k):
                env = get_environment(task.env_name)
                scorer = get_scorer(task.scorer_name)
                start = time.perf_counter()
                try:
                    trajectory = agent.act(task, env)
                    passed = scorer(trajectory, task)
                    results.append(TaskResult(task.id, attempt, passed, trajectory, None, time.perf_counter() - start))
                except Exception as e:
                    results.append(TaskResult(task.id, attempt, False, None, str(e), time.perf_counter() - start))
        return results

print("AsyncBenchmarkRunner ready: run_sync() [baseline] and run_async(agent, k, concurrency) [new]")


## Timing comparison: sequential vs. `concurrency=1` vs. `concurrency=8`

An 8-task suite × `k=3` attempts = 24 calls, each with a simulated 0.15s latency. Sequential should take roughly `24 × 0.15 ≈ 3.6s`. `concurrency=8` should take roughly `ceil(24/8) × 0.15 ≈ 0.45s` — an ~8x speedup, bounded by the semaphore, not by however many coroutines Python happens to schedule.


In [ ]:
tasks = [
    Task(id=f"calc-{i}", category="tool_use", prompt=f"Compute: 2+{i} = {2+i}", env_name="calc", scorer_name="exact_match")
    for i in range(4)
] + [
    Task(id=f"file-{i}", category="file_edit", prompt=f"Write task {i}'s output file", env_name="file", scorer_name="file_written")
    for i in range(4)
]

agent = MockAgent(flake_rate=0.0, latency=LATENCY)
runner = AsyncBenchmarkRunner(tasks)
K = 3

t0 = time.perf_counter()
sync_results = runner.run_sync(agent, k=K)
sync_elapsed = time.perf_counter() - t0

t0 = time.perf_counter()
async_c1_results = await runner.run_async(agent, k=K, concurrency=1)
async_c1_elapsed = time.perf_counter() - t0

t0 = time.perf_counter()
async_c8_results = await runner.run_async(agent, k=K, concurrency=8)
async_c8_elapsed = time.perf_counter() - t0

n_calls = len(tasks) * K
table = Table(title=f"{n_calls} calls @ {LATENCY}s simulated latency each")
table.add_column("Strategy"); table.add_column("Wall-clock (s)"); table.add_column("Speedup vs sequential")
table.add_row("run_sync() [baseline]", f"{sync_elapsed:.2f}", "1.0x")
table.add_row("run_async(concurrency=1)", f"{async_c1_elapsed:.2f}", f"{sync_elapsed/async_c1_elapsed:.1f}x")
table.add_row("run_async(concurrency=8)", f"{async_c8_elapsed:.2f}", f"{sync_elapsed/async_c8_elapsed:.1f}x")
console.print(table)

assert all(r.passed for r in sync_results), "sync baseline should 100%-pass a zero-flake MockAgent"
assert all(r.passed for r in async_c1_results), "concurrency=1 should match sync correctness exactly"
assert all(r.passed for r in async_c8_results), "concurrency=8 must not sacrifice correctness for speed"
assert async_c8_elapsed < sync_elapsed / 2, "expected concurrency=8 to be meaningfully faster than sequential"
print("\nAll correctness assertions passed. concurrency=8 was measurably faster without changing any scores.")


## Failure isolation: one flaky attempt shouldn't sink the batch

`_run_one` already wraps the agent call in `try/except`, converting a raised exception into a failed `TaskResult` rather than letting it propagate. This is the async-runner equivalent of L57's `batch_distill()` pitfall #4 (`asyncio.gather` without `return_exceptions=True` aborts the whole batch on the first failure). Let's prove it with a `MockAgent` that fails part of the time.


In [ ]:
flaky_agent = MockAgent(flake_rate=0.4, latency=0.02)
flaky_results = await runner.run_async(flaky_agent, k=5, concurrency=6)

n_failed = sum(1 for r in flaky_results if r.error is not None)
n_ok = sum(1 for r in flaky_results if r.error is None)

print(f"{len(flaky_results)} attempts submitted -> {n_ok} completed normally, {n_failed} caught as failed TaskResults")
assert n_failed > 0, "expected at least one simulated failure with flake_rate=0.4"
assert len(flaky_results) == len(tasks) * 5, "gather must return one result per submitted coroutine, failures included"
print("Confirmed: the whole batch finished even though ~40% of individual attempts raised.")


## Progress reporting during an async run

`asyncio.gather` gives you all results at once, at the end — no incremental feedback while a long suite is running. `asyncio.as_completed` yields each coroutine's result as soon as it finishes (not in submission order), which pairs naturally with a `rich.Progress` bar.


In [ ]:
async def run_async_with_progress(runner: AsyncBenchmarkRunner, agent, k: int, concurrency: int):
    sem = asyncio.Semaphore(concurrency)
    loop = asyncio.get_event_loop()
    coros = [runner._run_one(agent, task, a, sem, loop)
             for task in runner.tasks for a in range(k)]

    results = []
    with Progress(TextColumn("[bold]{task.description}"), BarColumn(), TextColumn("{task.completed}/{task.total}"),
                  TimeElapsedColumn(), console=console) as progress:
        pbar = progress.add_task("agent-bench suite", total=len(coros))
        for coro in asyncio.as_completed(coros):
            results.append(await coro)
            progress.update(pbar, advance=1)
    return results

progress_results = await run_async_with_progress(runner, agent, k=2, concurrency=4)
print(f"\n{len(progress_results)} results collected via as_completed (order not guaranteed to match submission order).")
assert len(progress_results) == len(tasks) * 2


## Module wiring: `agent_bench/runner_async.py` + CLI `--concurrency`

Writing the async runner to disk as its own module (imported by both `core`'s `BenchmarkRunner` and the CLI), plus a minimal Typer command exposing `--concurrency` so a user can tune it from the command line — same "thin CLI wrapper, zero business logic" rule from L58/L65.


In [ ]:
import pathlib

CONTENT_DIR = "/content/agent_bench"   # Colab path; validation runs substitute this, see note below
pathlib.Path(CONTENT_DIR).mkdir(parents=True, exist_ok=True)

runner_async_src = '''\"\"\"agent_bench/runner_async.py -- Semaphore-bounded concurrent benchmark execution.

Added in Lesson 67. Environments MUST be registered as classes (see registry.py),
never as shared instances -- get_environment() instantiates fresh per attempt so
concurrent attempts can never contaminate each other's state.
\"\"\"
import asyncio
import time
from .core import Task, TaskResult
from .registry import get_environment, get_scorer


class AsyncBenchmarkRunner:
    def __init__(self, tasks: list[Task]):
        self.tasks = tasks

    async def _run_one(self, agent, task: Task, attempt: int, sem: asyncio.Semaphore, loop) -> TaskResult:
        async with sem:
            env = get_environment(task.env_name)
            scorer = get_scorer(task.scorer_name)
            start = time.perf_counter()
            try:
                trajectory = await loop.run_in_executor(None, agent.act, task, env)
                passed = scorer(trajectory, task)
                return TaskResult(task.id, attempt, passed, trajectory, None, time.perf_counter() - start)
            except Exception as e:
                return TaskResult(task.id, attempt, False, None, str(e), time.perf_counter() - start)

    async def run_async(self, agent, k: int = 1, concurrency: int = 4) -> list[TaskResult]:
        sem = asyncio.Semaphore(concurrency)
        loop = asyncio.get_event_loop()
        coros = [self._run_one(agent, task, a, sem, loop)
                 for task in self.tasks for a in range(k)]
        return await asyncio.gather(*coros)
'''

with open(f"{CONTENT_DIR}/runner_async.py", "w") as f:
    f.write(runner_async_src)

print(f"Wrote {CONTENT_DIR}/runner_async.py ({len(runner_async_src)} bytes)")


In [ ]:
import typer
from typer.testing import CliRunner

cli_app = typer.Typer(rich_markup_mode="rich")

@cli_app.command("run")
def cmd_run(
    concurrency: int = typer.Option(4, "--concurrency", "-c", help="Max attempts running at once"),
    k: int = typer.Option(1, "--k", help="Attempts per task (for pass@k)"),
):
    # Run the benchmark suite, k attempts per task, up to --concurrency at once.
    results = asyncio.run(runner.run_async(agent, k=k, concurrency=concurrency))
    passed = sum(1 for r in results if r.passed)
    typer.echo(f"{passed}/{len(results)} passed (concurrency={concurrency}, k={k})")
    if passed < len(results):
        raise typer.Exit(code=1)

cli_runner = CliRunner()
# NOTE: a Typer app with exactly one registered command collapses to that command
# directly -- invoke as `--concurrency 8 --k 2`, NOT `run --concurrency 8 --k 2`.
# (A real multi-command agent-bench CLI, per L58/L65, keeps subcommand names; this
# minimal single-command demo app does not.)
result = cli_runner.invoke(cli_app, ["--concurrency", "8", "--k", "2"])
print(result.stdout)
assert result.exit_code == 0, f"expected clean exit on a zero-flake MockAgent, got {result.exit_code}"
print("CLI smoke test passed: `agent-bench --concurrency 8 --k 2` exits 0 on a passing suite.")


## Pitfalls

| # | Pitfall | Why it bites |
|---|---|---|
| 1 | Registering environments as shared **instances** instead of classes | The exact bug reproduced above — concurrent attempts silently contaminate each other's state; scores become meaningless without ever raising an error |
| 2 | No `Semaphore` at all (`gather` on hundreds of raw coroutines) | Thundering-herd against a real API — hits provider rate limits (429s), which look like agent failures in your results, not infra failures |
| 3 | `asyncio.gather(...)` without per-attempt `try/except` or `return_exceptions=True` | One flaky attempt aborts the entire suite (L57 pitfall #4, now hitting the benchmark harness itself) |
| 4 | Assuming `run_in_executor(None, ...)` parallelism helps CPU-bound work | The default executor is a **thread pool** — great for I/O-bound blocking calls (network, disk), useless for CPU-bound work under the GIL. A real scorer doing heavy local computation won't speed up this way |
| 5 | `asyncio.as_completed` result order == submission order | It isn't — results arrive in *completion* order. Don't index into them positionally; carry `task_id`/`attempt` in the result itself (already done above) |
| 6 | Shared mutable counters (e.g. running cost total) updated from concurrent coroutines without care | Python's GIL makes `+=` on a simple counter "safe enough" in practice for asyncio (single-threaded event loop), but the moment you mix in real threads (as `run_in_executor` does), that safety assumption weakens — prefer returning per-attempt values and summing at the end, as done here |
| 7 | No per-attempt timeout | One hung network call (real Claude API, real network) blocks that semaphore slot forever; wrap the executor call in `asyncio.wait_for(..., timeout=N)` in production |
| 8 | Forgetting `nest_asyncio.apply()` in a notebook | Jupyter/Colab already runs an event loop; top-level `await` and `asyncio.run()` inside notebook cells raise `RuntimeError: asyncio.run() cannot be called from a running event loop` without it |
| 9 | Concurrency knob set higher than your API tier's actual RPM limit | Local speedup, but the provider throttles you anyway — tune `--concurrency` against your real rate limit, not just wall-clock curiosity |
| 10 | Confusing `concurrency` (this lesson) with `k` (pass@k, L61) | They compose (`k` attempts per task, up to `concurrency` running at once) but solve different problems — `k` measures capability under repeated sampling, `concurrency` is purely an execution-speed knob and must never change *which* attempts run, only *how fast* |


In [ ]:
# Verification checklist

_names_defined = dir()  # snapshot BEFORE the comprehension -- dir() called inside a
                         # genexpr resolves to the genexpr's own local scope (an L65
                         # pitfall we're avoiding here, not repeating)

checks = {
    "core.py models recreated (Task/Trajectory/TaskResult/pass_at_k)": all(n in _names_defined for n in ["Task", "TrajectoryStep", "Trajectory", "TaskResult", "pass_at_k"]),
    "registry stores ENVIRONMENT classes, not instances": isinstance(ENVIRONMENT_REGISTRY["file"], type),
    "get_environment() returns a fresh instance each call": get_environment("file") is not get_environment("file"),
    "contamination bug reproduced (shared instance)": contaminated,
    "contamination fix verified (fresh instances)": all(list(state.keys()) == [f"task_{tid}.txt"] for tid, state in results_good),
    "AsyncBenchmarkRunner defined": "AsyncBenchmarkRunner" in dir(),
    "run_sync 100% pass on zero-flake agent": all(r.passed for r in sync_results),
    "run_async(concurrency=1) matches run_sync correctness": all(r.passed for r in async_c1_results),
    "run_async(concurrency=8) measurably faster than sequential": async_c8_elapsed < sync_elapsed / 2,
    "flaky agent failures isolated, not fatal": n_failed > 0 and len(flaky_results) == len(tasks) * 5,
    "as_completed progress run returned all results": len(progress_results) == len(tasks) * 2,
    "runner_async.py written to disk": os.path.exists(f"{CONTENT_DIR}/runner_async.py"),
    "CLI --concurrency smoke test exits 0": result.exit_code == 0,
}

table = Table(title="Lesson 67 Verification")
table.add_column("Check"); table.add_column("Status")
all_pass = True
for name, ok in checks.items():
    table.add_row(name, "PASS" if ok else "FAIL")
    all_pass = all_pass and ok
console.print(table)
assert all_pass, "one or more verification checks failed"
print("\nAll checks passed." if all_pass else "\nSome checks failed - review above.")


## Summary

| Concept | What it is |
|---|---|
| `AsyncBenchmarkRunner.run_async(agent, k, concurrency)` | Bounded-concurrency replacement for the sequential `BenchmarkRunner.run()` |
| `asyncio.Semaphore(concurrency)` | Caps how many attempts run "at once", protecting against thundering-herd / rate limits |
| `loop.run_in_executor(None, sync_fn, ...)` | Offloads a blocking sync call (agent.act, a real HTTP round-trip) onto a thread so it doesn't block the event loop |
| Environments registered as **classes**, not instances | `get_environment(name)` returns a fresh object per call — the concurrency-safety fix, demonstrated as a real bug-then-fix |
| Per-attempt `try/except` inside `_run_one` | Isolates a flaky attempt's failure into a `TaskResult`, so `gather()` never aborts the whole suite |
| `asyncio.as_completed` + `rich.Progress` | Incremental progress feedback during a long concurrent run, at the cost of losing submission order |
| CLI `--concurrency` flag | User-tunable knob, thin wrapper calling `asyncio.run(runner.run_async(...))` |

### Homework

1. Add a per-attempt timeout via `asyncio.wait_for(loop.run_in_executor(...), timeout=N)` and a `TaskResult` variant for timeout vs. error.
2. Extend the CLI with `--concurrency auto` that picks a default from `os.cpu_count()` or a configured API-tier RPM limit.
3. Reproduce the same contamination bug for `CalcEnv` (shared `history` list) instead of `FileEnv`, to confirm it's a general registry-design issue, not FileEnv-specific.
4. Add a real cost-tracking accumulator across concurrent attempts and reason about whether it needs a lock (hint: pure-Python `asyncio` code is single-threaded at the event-loop level, but `run_in_executor` callbacks run on real OS threads — where does the `+=` actually happen?).
5. Wire `run_async` into the real `ClaudeToolAgent` from L61 and measure the speedup against a live API (mind your rate limits — start with `concurrency=2`).

### L68 preview

CLI polish + PyPI packaging pass for `agent-bench` itself — turning today's `--concurrency` flag and everything from L65/L66 into a real installable, versioned package with an entry-point console script, following the same playbook L58 used for `paper-distiller`.
